In [0]:

-- 1. Read, parse, union, type, trim (still incremental, appends once)
CREATE TEMPORARY STREAMING TABLE stg_transactions_typed AS
WITH offline_stream AS (
  SELECT
    TRIM(transaction_date) AS raw_transaction_date,
    TRIM(stock_date) AS raw_stock_date,
    TRIM(product_id) AS raw_product_id,
    TRIM(customer_id) AS raw_customer_id,
    TRIM(store_id) AS raw_store_id,
    TRIM(quantity) AS raw_quantity,
    TRIM(transaction_mode) AS transaction_mode,
    ingested_timestamp
  FROM STREAM(mavenmarket.bronze.transactions)
),
online_parsed AS (
  SELECT
    from_json(
      message_payload,
      'transaction_date STRING, stock_date STRING, product_id STRING, customer_id STRING, store_id STRING, quantity STRING'
    ) AS payload,
    event_timestamp AS ingested_timestamp
  FROM STREAM(mavenmarket.bronze.kafka_online_orders)
),
online_stream AS (
  SELECT
    TRIM(payload.transaction_date) AS raw_transaction_date,
    TRIM(payload.stock_date) AS raw_stock_date,
    TRIM(payload.product_id) AS raw_product_id,
    TRIM(payload.customer_id) AS raw_customer_id,
    TRIM(payload.store_id) AS raw_store_id,
    TRIM(payload.quantity) AS raw_quantity,
    'online' AS transaction_mode,
    ingested_timestamp
  FROM online_parsed
),
combined_transactions AS (
  SELECT * FROM offline_stream
  UNION ALL
  SELECT * FROM online_stream
)
SELECT
  md5(concat_ws('|', raw_transaction_date, raw_product_id, raw_customer_id,
                raw_store_id, raw_quantity, CAST(ingested_timestamp AS STRING))) AS transaction_sk,

  COALESCE(
    try_to_date(raw_transaction_date, 'M/d/yyyy'),
    try_to_date(raw_transaction_date, 'yyyy-M-d'),
    try_to_date(raw_transaction_date, 'd-MMM-yy'),
    try_cast(raw_transaction_date AS DATE)
  ) AS transaction_date,

  COALESCE(
    try_to_date(raw_stock_date, 'M/d/yyyy'),
    try_to_date(raw_stock_date, 'yyyy-M-d'),
    try_to_date(raw_stock_date, 'd-MMM-yy'),
    try_cast(raw_stock_date AS DATE)
  ) AS stock_date,

  -- DECIMAL hop tolerates keys like '123.0'
  try_cast(try_cast(raw_product_id AS DECIMAL(18,0)) AS INT) AS product_id,
  try_cast(try_cast(raw_customer_id AS DECIMAL(18,0)) AS INT) AS customer_id,
  try_cast(try_cast(raw_store_id AS DECIMAL(18,0)) AS INT) AS store_id,
  try_cast(try_cast(raw_quantity AS DECIMAL(18,0)) AS INT) AS quantity,
  transaction_mode,

  -- Optional link to promotions, deterministic (0 to 19, or NULL)
  CASE
    WHEN pmod(hash(raw_customer_id, raw_product_id, ingested_timestamp), 10) >= 3
    THEN pmod(hash(raw_product_id, raw_store_id, ingested_timestamp), 20)
  END AS promotion_id,

  ingested_timestamp
FROM combined_transactions;

-- 2. Dimension lookup views.
--    as-of view: first version backdated so historical transactions match.
--    latest view: fallback when no version covers the transaction date.
CREATE TEMPORARY VIEW v_customer_asof AS
SELECT customer_id, customer_sk,
  CASE WHEN row_number() OVER (PARTITION BY customer_id ORDER BY __START_AT) = 1
       THEN DATE'1900-01-01' ELSE CAST(__START_AT AS DATE) END AS valid_from,
  COALESCE(CAST(__END_AT AS DATE), DATE'9999-12-31') AS valid_to
FROM LIVE.silver_customers;

CREATE TEMPORARY VIEW v_customer_latest AS
SELECT customer_id, customer_sk FROM LIVE.silver_customers
QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY __START_AT DESC) = 1;

CREATE TEMPORARY VIEW v_product_asof AS
SELECT product_id, product_sk,
  CASE WHEN row_number() OVER (PARTITION BY product_id ORDER BY __START_AT) = 1
       THEN DATE'1900-01-01' ELSE CAST(__START_AT AS DATE) END AS valid_from,
  COALESCE(CAST(__END_AT AS DATE), DATE'9999-12-31') AS valid_to
FROM LIVE.silver_products;

CREATE TEMPORARY VIEW v_product_latest AS
SELECT product_id, product_sk FROM LIVE.silver_products
QUALIFY row_number() OVER (PARTITION BY product_id ORDER BY __START_AT DESC) = 1;

CREATE TEMPORARY VIEW v_store_asof AS
SELECT store_id, store_sk,
  CASE WHEN row_number() OVER (PARTITION BY store_id ORDER BY __START_AT) = 1
       THEN DATE'1900-01-01' ELSE CAST(__START_AT AS DATE) END AS valid_from,
  COALESCE(CAST(__END_AT AS DATE), DATE'9999-12-31') AS valid_to
FROM LIVE.silver_stores;

CREATE TEMPORARY VIEW v_store_latest AS
SELECT store_id, store_sk FROM LIVE.silver_stores
QUALIFY row_number() OVER (PARTITION BY store_id ORDER BY __START_AT DESC) = 1;

CREATE TEMPORARY VIEW v_promotion_latest AS
SELECT promotion_id, promotion_sk FROM LIVE.silver_promotions
QUALIFY row_number() OVER (PARTITION BY promotion_id ORDER BY __START_AT DESC) = 1;

-- 3. Checks and surrogate key lookups (batch view, recomputed each update)
CREATE TEMPORARY VIEW stg_transactions_checked AS
SELECT
  t.*,
  COALESCE(pa.product_sk, pl.product_sk) AS product_sk,
  COALESCE(ca.customer_sk, cl.customer_sk) AS customer_sk,
  COALESCE(sa.store_sk, sl.store_sk) AS store_sk,
  prl.promotion_sk AS promotion_sk,

  -- Hard checks
  CASE WHEN t.transaction_date IS NULL THEN 'Completeness: transaction_date is null; ' ELSE '' END AS err_date_null,
  CASE WHEN t.quantity IS NULL THEN 'Completeness: quantity is null; ' ELSE '' END AS err_qty_null,
  CASE WHEN t.quantity <= 0 THEN 'Validity: quantity must be greater than 0; ' ELSE '' END AS err_qty_neg,
  CASE WHEN t.product_id IS NULL THEN 'Completeness: product_id is null; ' ELSE '' END AS err_product_null,
  CASE WHEN t.customer_id IS NULL THEN 'Completeness: customer_id is null; ' ELSE '' END AS err_customer_null,
  CASE WHEN t.store_id IS NULL THEN 'Completeness: store_id is null; ' ELSE '' END AS err_store_null,
  CASE WHEN t.transaction_mode IS NULL THEN 'Completeness: transaction_mode is null; ' ELSE '' END AS err_mode_null,

  -- Referential integrity: null only if the key exists in no version at all
  CASE WHEN t.product_id IS NOT NULL AND COALESCE(pa.product_sk, pl.product_sk) IS NULL
       THEN 'Referential Integrity: product_id not found in silver_products; ' ELSE '' END AS err_fk_product,
  CASE WHEN t.customer_id IS NOT NULL AND COALESCE(ca.customer_sk, cl.customer_sk) IS NULL
       THEN 'Referential Integrity: customer_id not found in silver_customers; ' ELSE '' END AS err_fk_customer,
  CASE WHEN t.store_id IS NOT NULL AND COALESCE(sa.store_sk, sl.store_sk) IS NULL
       THEN 'Referential Integrity: store_id not found in silver_stores; ' ELSE '' END AS err_fk_store,

  -- Warn checks (row stays in Silver)
  CASE WHEN t.promotion_id IS NOT NULL AND prl.promotion_sk IS NULL
       THEN 'Referential Integrity: promotion_id not found; ' ELSE '' END AS warn_fk_promo,
  CASE WHEN (ca.customer_sk IS NULL AND cl.customer_sk IS NOT NULL)
         OR (pa.product_sk IS NULL AND pl.product_sk IS NOT NULL)
       THEN 'Consistency: no dimension version at transaction_date, used latest; ' ELSE '' END AS warn_dim_fallback,
  CASE WHEN t.stock_date IS NULL THEN 'Completeness: stock_date is null; ' ELSE '' END AS warn_stock_date,
  CASE WHEN t.transaction_date < t.stock_date THEN 'Consistency: transaction_date is before stock_date; ' ELSE '' END AS warn_date_logic
FROM LIVE.stg_transactions_typed t
LEFT JOIN LIVE.v_product_asof pa
  ON t.product_id = pa.product_id
  AND t.transaction_date >= pa.valid_from AND t.transaction_date < pa.valid_to
LEFT JOIN LIVE.v_product_latest pl ON t.product_id = pl.product_id
LEFT JOIN LIVE.v_customer_asof ca
  ON t.customer_id = ca.customer_id
  AND t.transaction_date >= ca.valid_from AND t.transaction_date < ca.valid_to
LEFT JOIN LIVE.v_customer_latest cl ON t.customer_id = cl.customer_id
LEFT JOIN LIVE.v_store_asof sa
  ON t.store_id = sa.store_id
  AND t.transaction_date >= sa.valid_from AND t.transaction_date < sa.valid_to
LEFT JOIN LIVE.v_store_latest sl ON t.store_id = sl.store_id
LEFT JOIN LIVE.v_promotion_latest prl ON t.promotion_id = prl.promotion_id;

-- 4. Consolidate flags
CREATE TEMPORARY VIEW stg_transactions_flagged AS
SELECT
  *,
  TRIM(CONCAT(err_date_null, err_qty_null, err_qty_neg, err_product_null, err_customer_null,
              err_store_null, err_mode_null, err_fk_product, err_fk_customer, err_fk_store)) AS hard_failures,
  TRIM(CONCAT(warn_fk_promo, warn_dim_fallback, warn_stock_date, warn_date_logic)) AS dq_flags
FROM LIVE.stg_transactions_checked;

-- 5. Valid records (materialized view: recomputed against current dimensions)
CREATE MATERIALIZED VIEW silver_transactions
COMMENT "Cleaned transactions with surrogate keys and warn-level data quality flags."
AS SELECT
  * EXCEPT(err_date_null, err_qty_null, err_qty_neg, err_product_null, err_customer_null,
           err_store_null, err_mode_null, err_fk_product, err_fk_customer, err_fk_store,
           warn_fk_promo, warn_dim_fallback, warn_stock_date, warn_date_logic, hard_failures)
FROM LIVE.stg_transactions_flagged
WHERE hard_failures = '';

-- 6. Quarantine (rows leave automatically once the missing dimension row arrives)
CREATE MATERIALIZED VIEW quarantine_transactions
COMMENT "Transactions failing hard checks, re-evaluated every update."
AS SELECT
  * EXCEPT(err_date_null, err_qty_null, err_qty_neg, err_product_null, err_customer_null,
           err_store_null, err_mode_null, err_fk_product, err_fk_customer, err_fk_store,
           warn_fk_promo, warn_dim_fallback, warn_stock_date, warn_date_logic, dq_flags),
  hard_failures AS quarantine_reasons
FROM LIVE.stg_transactions_flagged
WHERE hard_failures != '';